# 🚀 HappyGen Studio - Google Colab 16GB Cloud GPU Server
**Free Tesla T4 / A100 GPU • Full API: txt2img, img2img, interrogate, upscale, facefix**

### Instructions:
1. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
2. Run all cells below.
3. Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. In HappyGen Web App, click the **Settings / Backend** icon in the header, paste the URL, and click **Test Connection**!

In [1]:
# Cell 1: Install Dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf fastapi uvicorn pydantic pycloudflared nest_asyncio python-multipart peft open_clip_torch
!pip install -q git+https://github.com/xinntao/BasicSR.git
!pip install -q git+https://github.com/xinntao/Real-ESRGAN.git
!pip install -q gfpgan


In [2]:
# Cell 2: Download Models & SDXL Lightning Accelerator
import os
os.makedirs("/content/Models", exist_ok=True)
os.makedirs("/content/LoRAs", exist_ok=True)
os.makedirs("/content/Embeddings", exist_ok=True)

# Define the path for the desired Civitai base model
BASE_MODEL_PATH = "/content/Models/crucibleRINGPonyxl_v28.safetensors"
LIGHTNING_PATH = "/content/LoRAs/sdxl_lightning_4step_lora.safetensors"

# Import userdata for secrets
from google.colab import userdata

try:
    CIVITAI_API_KEY = userdata.get('CIVITAI_API_KEY')
except Exception:
    CIVITAI_API_KEY = None

def civitai_download_url(model_version_id):
    base_url = f"https://civitai.com/api/download/models/{model_version_id}"
    if CIVITAI_API_KEY:
        return f"{base_url}?token={CIVITAI_API_KEY}"
    return base_url

# Clean up corrupted files
if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
    print(f"⚠️ Found corrupted or incomplete file at {BASE_MODEL_PATH}. Deleting and re-downloading...")
    os.remove(BASE_MODEL_PATH)

if not os.path.exists(BASE_MODEL_PATH):
    print("📥 Downloading CrucibleRING PonyXL v28 (~6.6GB) from Civitai...")
    !wget -c "{civitai_download_url('1979291')}" -O {BASE_MODEL_PATH}
    if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
        print(f"⚠️ Download failed. It might be corrupted. Deleting file.")
        os.remove(BASE_MODEL_PATH)
    else:
        print(f"✅ Base model downloaded successfully.")

if not os.path.exists(LIGHTNING_PATH):
    print("⚡ Downloading SDXL Lightning 4-Step LoRA...")
    !wget -c "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_lora.safetensors" -O {LIGHTNING_PATH}

print("✅ Storage ready! Models directory is prepared.")


In [3]:
# Cell 3: Load Model into 16GB Cloud VRAM
import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler

print("🧹 Cleaning up /content/Models from non-base model safetensors files...")
for filename in os.listdir("/content/Models"):
    file_path = os.path.join("/content/Models", filename)
    if filename.endswith(".safetensors") and file_path != BASE_MODEL_PATH:
        print(f"    Deleting file: {file_path}")
        os.remove(file_path)

print(f"🚀 Initializing Pipeline on {torch.cuda.get_device_name(0)}...")
CURRENT_BASE_MODEL_FILE = os.path.basename(BASE_MODEL_PATH)
global pipe
global pipe_img2img

pipe = StableDiffusionXLPipeline.from_single_file(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

pipe_img2img = StableDiffusionXLImg2ImgPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    text_encoder_2=pipe.text_encoder_2,
    tokenizer=pipe.tokenizer,
    tokenizer_2=pipe.tokenizer_2,
    unet=pipe.unet,
    scheduler=pipe.scheduler,
)

print("✅ Base model loaded successfully and ready for dynamic UI injection.")


In [4]:
# Cell 4: Launch FastAPI Server & Cloudflare Public Tunnel
import io, base64, time, json, threading, nest_asyncio, os, uuid
import numpy as np
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Union
import uvicorn
from pycloudflared import try_cloudflare
import requests
import gc
import subprocess

nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ============================================================================
# Lazy-loaded utility models (loaded on first use, unloaded after to save VRAM)
# ============================================================================
_clip_model = None
_clip_preprocess = None
_upscaler = None
_face_restorer = None

tasks = {}
def _bg_runner(task_id, func, *args, **kwargs):
    try:
        res = func(*args, **kwargs)
        tasks[task_id]['status'] = 'completed'
        tasks[task_id]['result'] = res
    except Exception as e:
        tasks[task_id]['status'] = 'failed'
        tasks[task_id]['error'] = str(e)

@app.get("/async/status/{task_id}")
def async_status(task_id: str):
    if task_id not in tasks: return {"status": "not_found"}
    return tasks[task_id]


_tagger_session = None
_tagger_tags = None

def _load_tagger():
    global _tagger_session, _tagger_tags
    if _tagger_session is not None: return
    import subprocess
    import csv
    print("📦 Installing WD14 Tagger dependencies...")
    subprocess.check_call(["pip", "install", "-q", "onnxruntime-gpu", "huggingface_hub"])
    
    from huggingface_hub import hf_hub_download
    import onnxruntime as ort
    
    repo_id = "SmilingWolf/wd-v1-4-moat-tagger-v2"
    print("🔍 Downloading WD14 Tagger Model...")
    model_path = hf_hub_download(repo_id, "model.onnx")
    tags_path = hf_hub_download(repo_id, "selected_tags.csv")
    
    with open(tags_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        _tagger_tags = [row[1] for row in reader]
        
    _tagger_session = ort.InferenceSession(model_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

def _unload_tagger():
    global _tagger_session, _tagger_tags
    if _tagger_session is not None:
        del _tagger_session
        _tagger_session = None
        _tagger_tags = None
        gc.collect()
        torch.cuda.empty_cache()

def _do_interrogate(req: InterrogateRequest):
    _load_tagger()
    image = _decode_base64_image(req.image)
    
    target_size = 448
    img = image.convert("RGB")
    w, h = img.size
    max_dim = max(w, h)
    pad_img = Image.new("RGB", (max_dim, max_dim), (255, 255, 255))
    pad_img.paste(img, (max_dim//2 - w//2, max_dim//2 - h//2))
    img = pad_img.resize((target_size, target_size), Image.BICUBIC)
    
    image_array = np.array(img, dtype=np.float32)
    image_array = image_array[:, :, ::-1] # RGB to BGR
    image_array = np.expand_dims(image_array, axis=0)
    
    input_name = _tagger_session.get_inputs()[0].name
    output_name = _tagger_session.get_outputs()[0].name
    
    preds = _tagger_session.run([output_name], {input_name: image_array})[0][0]
    
    threshold = 0.35
    tag_preds = preds[4:]
    tag_names = _tagger_tags[4:]
    
    final_tags = []
    for p, name in zip(tag_preds, tag_names):
        if p > threshold:
            final_tags.append(name.replace('_', ' '))
            
    _unload_tagger()
    return {"caption": ", ".join(final_tags)}


@app.post("/sdapi/v1/interrogate")
def interrogate(req: InterrogateRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_interrogate, req)).start()
    return {"task_id": task_id}

def _do_upscale(req: UpscaleRequest):
    _load_upscaler()
    image = _decode_base64_image(req.image)
    img_bgr = np.array(image)[:, :, ::-1]
    output, _ = _upscaler.enhance(img_bgr, outscale=req.upscaling_resize)
    result_image = Image.fromarray(output[:, :, ::-1])
    _unload_upscaler()
    return {"images": [_encode_image_to_base64(result_image)], "source": "Real-ESRGAN"}

@app.post("/sdapi/v1/extra-single-image")
def upscale(req: UpscaleRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_upscale, req)).start()
    return {"task_id": task_id}

def _do_face_fix(req: FaceFixRequest):
    _load_face_restorer()
    image = _decode_base64_image(req.image)
    img_bgr = np.array(image)[:, :, ::-1]
    _, _, output = _face_restorer.enhance(img_bgr, has_aligned=False, only_center_face=False, paste_back=True)
    result_image = Image.fromarray(output[:, :, ::-1])
    _unload_face_restorer()
    return {"images": [_encode_image_to_base64(result_image)], "source": "GFPGAN"}

@app.post("/sdapi/v1/face-fix")
def face_fix(req: FaceFixRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_face_fix, req)).start()
    return {"task_id": task_id}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)
tunnel = try_cloudflare(port=8000)
print(f"\n🎉 COPY THIS URL: {tunnel.tunnel}\n")
